Copyright 2025 DeepMind Technologies Limited.

In [ ]:
# @title SPDX-License-Identifier: Apache-2.0
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#      http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# RAG Tutorial with OneTwo

This tutorial demonstrates how to build a **Retrieval-Augmented Generation
(RAG)** system using the OneTwo framework.

## What is RAG?

At its core, **RAG** is a technique for answering queries that require knowledge
not present in the LLM's training data (such as private corpora or information
post-dating the model's knowledge cutoff). It works by:

1. **Retrieving** query-relevant information spread across multiple documents in an external source (e.g., a database or the web).
2. **Augmenting** the LLM's context with the retrieved content.
3. **Generating** a grounded answer using the augmented context.

![Retrieval Augmented Generation (RAG) Flow](images/rag.png)

## Focus of this Tutorial: RAG on Private Corpora

While RAG can be used to pull live search results from the web, **this tutorial
focuses on RAG for private corpora** (such as proprietary company wikis, private
document collections, or domain-specific files).

To search through a private corpus, we need to structure the raw corpus first.
This is where an **Index** comes in. You can think of an **Index** as a
specialized database of your documents, optimized for fast semantic search. In
OneTwo, the **Index acts directly as the Retriever** in our RAG pipeline. The
conceptual swap looks like this:

![Index as the Retriever in OneTwo](images/index_as_retriever.png)

> **Note:** If you are wondering how we construct the Index in the first place,
> don't worry! Building the Index is a key step that we will walk you through in
> detail in the **Index Building** section later.

In this tutorial, we will use OneTwo's standard, flexible interfaces to build
the RAG-on-private-corpora system from scratch. We will cover:
1. **Basic Retrieval**: Understanding the `Retriever` interface.
2. **Index Building**: Preparing proprietary documents and building highly performant semantic indexes.
3. **Question Answering Strategies**: Combining retrieval and generation for
robust Question Answering.
4. **QA Strategies on Real-World Datasets**:
Performing RAG on HotpotQA and evaluating its quality.
5. **OneTwo RAG in the Agent Landscape**: A quick look at how OneTwo's RAG can be plugged into autonomous Agents as a Tool.

This tutorial is similar in spirit to the main [tutorial.ipynb](tutorial.ipynb)
focusing specifically on the RAG features of OneTwo.

In [1]:
# @title Boilerplate for conditional cell execution.
import os
import IPython


@IPython.core.magic.register_cell_magic
def skip_if(line, cell):
  if not eval(line):
    IPython.get_ipython().run_cell(cell)

## ⚡ Quick Start: RAG in 2 Lines

With OneTwo's top-level `rag` module, you can build an index over your documents and evaluate RAG quality in just **two function calls**:

In [2]:
%%skip_if os.environ.get('COLAB_NOTEBOOK_TEST')

import pprint
from onetwo import rag
from onetwo.evaluation import datasets

google_api_key = ''

# 1. Build an embedding index over your corpus.
index_dir = rag.build_index(
    corpus=datasets.get_dataset_loader(
        'hotpotqa_validation', max_number_of_documents=20
    ),
    output='/tmp/onetwo_rag_quickstart_index',
    runtime_kwargs={'api_key': google_api_key},
)

# 2. Evaluate RAG performance with AI raters and retrieval metrics.
summary = rag.evaluate(
    examples=datasets.get_dataset_loader(
        'hotpotqa_validation', max_number_of_examples=5
    ),
    index=index_dir,
    output='/tmp/onetwo_rag_quickstart_eval',
    runtime_kwargs={'api_key': google_api_key},
)
pprint.pprint(summary.metrics)

Loading dataset hotpotqa_validation...
Loaded 20 documents.
Will save the index to output directory: /tmp/onetwo_rag_quickstart_index/hotpotqa_validation/EmbeddingBasedDocumentIndex_chunk_strategy_words_chunk_size_None_overlap_window_0_gemini-embedding-2__hash_09ebd227
Building index...


Embedding docs: 100%|██████████| 20/20 [00:06<00:00,  2.89it/s]


Index built and saved to /tmp/onetwo_rag_quickstart_index/hotpotqa_validation/EmbeddingBasedDocumentIndex_chunk_strategy_words_chunk_size_None_overlap_window_0_gemini-embedding-2__hash_09ebd227.
Loading dataset hotpotqa_validation...
Loaded 5 examples.
Running evaluation...


100%|██████████| 5/5 [01:21<00:00, 16.22s/it]

Evaluation Metrics: {'tool_details': 0.0, 'accuracy': 0.8, 'recall@20': 0.4, 'total_output_tokens': 663.0, 'recall@5': 0.4, 'max_input_tokens': 2473.2, 'average_input_tokens': 2473.2, 'total_input_tokens': 2473.2, 'num_llm_calls': 1.0, 'max_output_tokens': 663.0, 'average_output_tokens': 663.0, 'num_tool_calls': 0.0}
HTML report saved to /tmp/onetwo_rag_quickstart_eval/results.html
{'tool_details': 0.0, 'accuracy': 0.8, 'recall@20': 0.4, 'total_output_tokens': 663.0, 'recall@5': 0.4, 'max_input_tokens': 2473.2, 'average_input_tokens': 2473.2, 'total_input_tokens': 2473.2, 'num_llm_calls': 1.0, 'max_output_tokens': 663.0, 'average_output_tokens': 663.0, 'num_tool_calls': 0.0}


> *Want to understand what's happening under the hood? Read on!*

# Setup

At minimum, these are the core libraries needed to initialize OneTwo and perform basic operations. Additional modules are imported throughout the tutorial close to where they are used.

In [ ]:
import copy
import dataclasses
import os
import pprint
import shutil
import sys

from onetwo import ot
from onetwo.builtins import llm

# Confirm the currently installed OneTwo version.
print(f'Running OneTwo v{ot.__version__}.')

## Connecting to a Model

We will use the `GoogleGenAIAPI` backend for this tutorial. If you don't have an API key, you can use Ollama models. Note that if you are running with Ollama models, the results might not be as expected in the captions of this notebook. For more details on
connecting to other backends (like `OpenAI`) or advanced configuration, please
refer to the main `tutorial.ipynb`.

In [ ]:
model_selection = 'Gemini API: gemini-3.1-flash-lite'  # @param ['Gemini API: gemini-3.1-flash-lite', 'Ollama API: gemma4:e2b']
embedder_selection = 'Gemini API: gemini-embedding-001'  # @param ['Gemini API: gemini-embedding-001', 'Ollama API: embeddinggemma']
tokenizer_model_name = (
    'https://huggingface.co/t5-small/resolve/main/spiece.model'
)

max_tokens_per_chunk = 12

### Authentication & Credentials

If using the Gemini API, obtain an API key from https://ai.google.dev/ and enter it below.

Alternatively, you can configure it via the `GOOGLE_API_KEY` environment variable or Colab user secrets, in which case you can leave the field below empty.

In [ ]:
# Gemini API key (from https://ai.google.dev/ or Colab secret / GOOGLE_API_KEY env)
google_api_key = ""  # @param {type: "string"}

### Backend Registration and Caching Configuration

This section configures request caching, credentials validation, and registers the model backends. These settings are automated and do not require user modification.

In [ ]:
# Resolve API key from form, Colab user secrets, or environment variable.
google_api_key = (
    google_api_key if 'google_api_key' in globals() and google_api_key else ''
)

if google_api_key:
  print('Using API key specified in the colab.')

try:
  from google.colab import userdata

  if not google_api_key:
    google_api_key = userdata.get('GOOGLE_API_KEY')
    print('Loaded GOOGLE_API_KEY from user secrets.')
except Exception:
  pass

if not google_api_key and 'GOOGLE_API_KEY' in os.environ:
  google_api_key = os.environ['GOOGLE_API_KEY']
  print('Loaded GOOGLE_API_KEY from environment.')

In [ ]:
if (
    model_selection.startswith('Gemini API')
    or embedder_selection.startswith('Gemini API')
) and not google_api_key:
  raise ValueError(
      'The GOOGLE_API_KEY must be specified either here or in a user secret or '
      'in the environment.'
  )

In [15]:
# Here we define a location in which to store a cache of requests/replies for
# each backend of interest, which we can use for speeding up running of the
# colab if we re-run it (or make iterative modifications to it) in the future.
# We will use a separate cache file for each backend.
OWN_CACHE_DIRECTORY = '/tmp/onetwo_colab_backend_caches/tutorial'

# If you would like to share cache files with others in your working group, you
# can optionally specify another shared cache directory here. If you specify
# this, then we will read from shared cache directory and give precedence to its
# contents, while merging in any additional content from OWN_CACHE_DIRECTORY.
# When saving the cache, however, we will by default write only to
# OWN_CACHE_DIRECTORY to reduce the chance of people clobbering each other's
# changes.
SHARED_CACHE_DIRECTORY = None

# If you want to automatically merge in content from any of your teammates'
# cache directories or from a cache that was output by a batch eval run,
# you can list the additional directories here.
ADDITIONAL_CACHE_DIRECTORIES = []

In [17]:
from onetwo.utils import colab_utils

# We encapsulate the logic for loading/saving the caches for a set of backends
# in the `CachedBackends` class. Each time we construct a backend, we will add
# it to the `cached_backends` mapping so that we can load and save them all at
# once.
cached_backends = colab_utils.CachedBackends(
    own_cache_directory=OWN_CACHE_DIRECTORY,
    shared_cache_directory=SHARED_CACHE_DIRECTORY,
    additional_cache_directories=ADDITIONAL_CACHE_DIRECTORIES,
)

In [18]:
from google.genai import types as genai_types
from onetwo.backends import google_genai_api
from onetwo.core import caching

# E.g., 'Gemini API: gemini-1.0-pro' ==> 'gemini-1.0-pro'.
model_name = ':'.join(model_selection.split(':')[1:]).strip()

if model_selection.startswith('Gemini API'):
  generate_config = {}
  # We are disabling safety settings for the purposes of this colab to
  # avoid the chance of getting empty responses in the case of
  # over-triggering of the safety controls. You can set this to an
  # appropriate setting for your use case.
  safety_settings = google_genai_api.SAFETY_DISABLED
  generate_config['safety_settings'] = safety_settings

  # For thinking models, you can control the amount of thinking budget
  # using the `ThinkingConfig` below. For the purposes of this colab,
  # we are disabling thinking by default, as it tends to not play well
  # with the use of the `max_tokens` parameter (i.e.,
  # `max_output_tokens` in the GoogleGenAI API), due to it getting
  # applied to the sum of the thinking tokens and actual output
  # tokens, which can often lead to empty outputs.
  thinking_budget = 128
  generate_config['thinking_config'] = genai_types.ThinkingConfig(
      include_thoughts=False, thinking_budget=thinking_budget
  )

  backend = google_genai_api.GoogleGenAIAPI(
      vertexai=False,
      api_key=google_api_key,
      temperature=0.0,
      generate_model_name=f'models/{model_name}',
      chat_model_name=f'models/{model_name}',
      threadpool_size=100,
      max_retries=3,
      cache=caching.SimpleFunctionCache(
          cache_filename=cached_backends.get_cache_path(model_name),
      ),
      generate_text_kwargs=copy.deepcopy(generate_config),
      generate_object_kwargs=copy.deepcopy(generate_config),
      generate_content_kwargs=copy.deepcopy(generate_config),
      chat_kwargs=copy.deepcopy(generate_config),
  )

  cached_backends[model_name] = backend
  backend.register(register_embed=False, register_tokenize=False)

  print(f'Gemini API backend registered: {model_name}')

elif model_selection.startswith('Ollama API'):
  from onetwo.backends import ollama_api

  backend = ollama_api.OllamaAPI(
      model_name=model_name,
      cache=caching.SimpleFunctionCache(
          cache_filename=cached_backends.get_cache_path(model_name),
      ),
  )

  cached_backends[model_name] = backend
  backend.register(register_embed=False)

  print(f'Ollama API backend registered: {model_name}')

else:
  raise ValueError(f'Unsupported model selection: {model_selection}')

Gemini API backend registered: gemini-3.1-flash-lite


In [19]:
# E.g., 'Gemini API: gemini-embedding-001' ==> 'gemini-embedding-001'.
embedder_model_name = embedder_selection.split(':')[-1].strip()

if embedder_selection.startswith('Gemini API'):
  embedder_backend = google_genai_api.GoogleGenAIAPI(
      vertexai=False,
      api_key=google_api_key,
      temperature=0.0,
      embed_model_name=f'models/{embedder_model_name}',
      threadpool_size=100,
      max_retries=3,
      cache=caching.SimpleFunctionCache(
          cache_filename=cached_backends.get_cache_path(embedder_model_name),
      ),
  )

  cached_backends[embedder_model_name] = embedder_backend
  embedder_backend.register(register_tokenize=False, register_generate=False)

  print(f'Gemini API backend for embed registered: {embedder_model_name}')

elif embedder_selection.startswith('Ollama API'):
  from onetwo.backends import ollama_api

  embedder_backend = ollama_api.OllamaAPI(
      embed_model_name=embedder_model_name,
      cache=caching.SimpleFunctionCache(
          cache_filename=cached_backends.get_cache_path(embedder_model_name),
      ),
  )

  cached_backends[embedder_model_name] = embedder_backend
  embedder_backend.register(register_generate=False)

  print(f'Ollama API backend for embed registered: {embedder_model_name}')

else:
  raise ValueError(f'Unsupported embedder selection: {embedder_selection}')

Gemini API backend for embed registered: gemini-embedding-001


In [20]:
from onetwo.backends import sentencepiece_tokenizer

tokenizer_backend = sentencepiece_tokenizer.SentencePieceTokenizer(
    model_name=tokenizer_model_name,
    cache_filename=cached_backends.get_cache_path(tokenizer_model_name),
)
tokenizer_backend.register()

print(f'Tokenizer backend registered: {tokenizer_model_name}')

Tokenizer backend registered: GEMMA


In [ ]:
cached_backends.load_caches()

# Basic Retrieval

In OneTwo, a `Retriever` is a generic interface that takes a query and returns a
list of relevant results (typically documents or text passages). A retriever can
be anything from a simple keyword search, a vector database, or even a web-based
search engine.

To showcase the interface, we will start by building a minimal mock retriever
(`SimpleDictRetriever`) and look at a web search retriever. Later, we will
replace these with a full private corpus Index.

In [22]:
from onetwo.stdlib.retrieval import retrieval


@dataclasses.dataclass
class SimpleDictRetriever(retrieval.Retriever[str, str]):
  data: dict[str, str] = dataclasses.field(default_factory=dict)

  @ot.make_executable(copy_self=False)
  async def retrieve(self, query: str, **kwargs) -> list[str]:
    return [self.data.get(query, "Not found")]


simple_data = {"France": "Paris", "Germany": "Berlin"}
mock_retriever = SimpleDictRetriever(data=simple_data)
result = ot.run(mock_retriever("France"))
print(f"Result: {result}")

Result: ['Paris']


> Note: We use `@ot.make_executable` to register custom classes as OneTwo
> executables (enabling features like caching) and call `ot.run()` to trigger
> their actual execution (see [tutorial.ipynb](tutorial.ipynb) for more
> details).

## Google Search as a Retriever

Below is an example of **Google Search** as a `Retriever`. As seen in
[tutorial.ipynb](tutorial.ipynb), one way to perform Google Search is leveraging
the tool-use capabilities of the
[Google Generative AI SDK](https://github.com/googleapis/python-genai).

NOTE: This example requires using the Google Search tool. It's not possible to
run this example with Ollama backends registered.

In [ ]:
from google.genai import types as genai_types
from onetwo.stdlib.retrieval import retrieval

genai_search_tool = genai_types.Tool(google_search=genai_types.GoogleSearch())
chat_config_kwargs = {
    'system_instruction': (
        'You are an expert in Google Search. Use the Search tool for every'
        ' query the user provides. Output the content without citations.'
    ),
    'tools': [genai_search_tool],
    'max_tokens': 128,
}


@dataclasses.dataclass
class SearchRetriever(retrieval.Retriever[str, str]):

  @ot.make_executable(copy_self=False)
  async def retrieve(
      self,
      query: str,
      *,
      max_results: int | None = None,
      **kwargs,
  ) -> list[str]:
    """Search engine. Returns a relevant snippet or answer to query."""
    result = await llm.instruct(query, **chat_config_kwargs)
    return [result]


search_as_retriever = SearchRetriever()

In [24]:
if model_selection.startswith("Gemini API"):
  query = "What is the capital of France?"
  result = ot.run(search_as_retriever(query))
  print(f"Result: {result}")

Result: ['The capital of France is Paris.']


# Index Building

For proprietary knowledge bases (such as private databases, or internal wikis),
we cannot simply perform a Google Search. Instead, we must build our own search
system. This workflow of processing your private documents and making them
searchable is called **Index Building**. OneTwo provides a highly modular and
flexible pipeline for building your Index, as visualized below:

![Index Building Pipeline](images/index_building.png)

The pipeline typically involves three main components: 1. **Corpus (Raw Docs)**:
Starting with your raw, unstructured document collection. 2. **Corpus Rewriter
(Chunking & Formatting)**: Processing the documents by breaking them into
smaller pieces (**Chunker**) and adding structure like titles (**Formatter**) so
the search is highly effective. 3. **Index**: Storing these processed documents
alongside their semantic vector embeddings.

### Understanding Corpus vs. Index

It is important to distinguish between these two core concepts:

*   The **Corpus** is simply your raw, unstructured data. On its own, it cannot
    be easily and effectively searched.

*   The **Index** is the structured representation of a corpus. By mapping
    documents (or chunks) to semantic **vector embeddings**, the Index
    transforms a static corpus into an active, searchable retrieval system.

In [ ]:
from onetwo.stdlib.retrieval import retrieval_data_structures

# Create a toy corpus of documents.
toy_corpus = [
    retrieval_data_structures.Document(
        doc_id="lion",
        content=(
            "Lions are large cats that live in prides. They are apex predators."
        ),
        title="Lion Facts",
    ),
    retrieval_data_structures.Document(
        doc_id="elephant",
        content=(
            "Elephants are the largest land animals. They have long trunks and"
            " tusks."
        ),
        title="Elephant Facts",
    ),
    retrieval_data_structures.Document(
        doc_id="monkey",
        content=(
            "Monkeys are primates that often live in trees. They are known for"
            " their intelligence."
        ),
        title="Monkey Facts",
    ),
]

## Indexing

In [26]:
from onetwo.stdlib.retrieval import indexing

toy_index = indexing.EmbeddingBasedDocumentIndex()

# `add_docs` will automatically calculate embeddings for all documents.
ot.run(toy_index.add_docs(toy_corpus))

print(f"Index created with {toy_index.num_docs} documents.")

Index created with 3 documents.


> While **Corpus Rewriting** is usually necessary for large, complex documents
> to ensure high-quality search results, it is entirely optional. If your
> documents are already short and clean like our `toy_corpus`, you can skip this
> step and build the Index directly from your raw Corpus as shown above.

Once the Index is built, **it acts directly as a Retriever** in the OneTwo RAG
paradigm. Let's test the retrieval using our created toy index.

In [27]:
query = "What is the largest land animal?"
results = ot.run(toy_index.retrieve(query, max_results=1))

print(f"Query: {query}")
for doc in results:
  print(f"Retrieved Doc ID: {doc.doc_id}")
  print(f"Content: {doc.content}")

Query: What is the largest land animal?
Retrieved Doc ID: elephant
Content: Elephants are the largest land animals. They have long trunks and tusks.


> NOTE: We set `max_results=1` to get the top-ranked result.

The LLM correctly retrieved the `elephant` document as the most relevant to the
query.

## Corpus Rewriting (Processing Large Documents)

In real-world applications, raw documents are rarely as short and clean as our
toy animal facts. They might be multi-page PDFs or long wiki articles. If we
index them directly, the semantic search will be inaccurate because the
embeddings will be too generic. To solve this, we use a **Corpus Rewriter** (as
shown in the middle block of the Index Building diagram) to **process** our
documents *before* they go into the index. We will demonstrate three key
rewriting operations:

1. **Chunking**: Breaking documents into smaller,
search-friendly pieces.
2. **Formatting**: Structuring the document content.
3. **Sequential Rewriting**: Chaining these operations together in sequence.

### Step A: Chunking (Breaking Documents Down)

We use a Chunker to split long, unstructured documents into smaller, digestible pieces. Limiting the amount of text in a single chunk (compared to a massive document) creates more precise semantic embeddings.

Depending on the document structure, various chunking strategies can be used —
such as splitting by structural boundaries (like paragraphs or sentences) or by
fixed character limits. However, since LLM inputs and costs
are measured in tokens, **token-based chunking** is the most common strategy in
production RAG systems.

OneTwo provides **`ChunkByMaxTokens`**, which splits documents
based on a maximum token count (`max_tokens_per_chunk`).

In [28]:
from onetwo.stdlib.retrieval import chunking
from onetwo.stdlib.retrieval import corpus_rewriting

max_tokens_chunker = chunking.ChunkByMaxTokens(
    max_tokens_per_chunk=max_tokens_per_chunk, overlap_window=1
)
chunking_rewriter = corpus_rewriting.ChunkingCorpusRewriter(
    chunker=max_tokens_chunker
)

chunked_docs = ot.run(chunking_rewriter(toy_corpus))

print(f"Original docs: {len(toy_corpus)}")
print(f"Chunked docs: {len(list(chunked_docs))}")
for doc in chunked_docs:
  print(f"Chunk ID: {doc.doc_id}, Content: {doc.content}")

Original docs: 3
Chunked docs: 6
Chunk ID: lion_1, Content: Lions are large cats that live in prides. They
Chunk ID: lion_2, Content: They are apex predators.
Chunk ID: elephant_1, Content: Elephants are the largest land animals. They have
Chunk ID: elephant_2, Content: have long trunks and tusks.
Chunk ID: monkey_1, Content: Monkeys are primates that often live in trees.
Chunk ID: monkey_2, Content: . They are known for their intelligence.


Look at the chunked outputs above — specifically **`Chunk ID: lion_2`**:

> *Content: "They are apex predators."*

OneTwo's `EmbeddingBasedDocumentIndex` uses the **content** field to calculate
search embeddings. In this setup, chunk `lion_2` presents a problem as the pronoun **"They"** has lost its context during the chunking process. The embedding model cannot generate a meaningful vector as it won't know what "They" refers to. To resolve contextual problems, one strategy is to use the **Formatter**.

### Formatting (Structuring Document Content)

The second step in our pipeline is the **Formatter**. Once documents are
chunked, they can lose context. A `DocumentFormatter` allows us to standardize
how documents are represented before embedding (e.g., by prefixing each chunk
with its document's `Title`).

Document formatting serves two purposes: (1) More precise semantic embeddings and (2) Meaningful Context for Generation as the retrieved document (chunk) forms the LLM context to generate a grounded answer.

We will use `TextDocumentFormatter` to prefix our animal facts with their
corresponding titles.

In [29]:
from onetwo.stdlib.retrieval import document_formatting

simple_formatter = document_formatting.TextDocumentFormatter(
    format_str="Title: {title}\nFacts: {text}",
)
formatting_rewriter = corpus_rewriting.FormattingCorpusRewriter(
    formatter=simple_formatter
)
formatted_docs = ot.run(formatting_rewriter(toy_corpus))

print(f"Original docs: {len(toy_corpus)}")
print(f"Formatted docs: {len(list(formatted_docs))}")
for doc in formatted_docs:
  print(f"Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

Original docs: 3
Formatted docs: 3
Doc ID: lion
Content:
Title: Lion Facts
Facts: Lions are large cats that live in prides. They are apex predators.
--------------------
Doc ID: elephant
Content:
Title: Elephant Facts
Facts: Elephants are the largest land animals. They have long trunks and tusks.
--------------------
Doc ID: monkey
Content:
Title: Monkey Facts
Facts: Monkeys are primates that often live in trees. They are known for their intelligence.
--------------------


### Sequential Rewriting

In real-world systems, you chain these steps together: first chunking the raw
docs, then formatting the resulting chunks. OneTwo provides
`SequentialCorpusRewriter` to build this unified Corpus Rewriter pipeline
(matching the entire yellow box in our diagram) and apply it to our corpus in
one go.

In [30]:
sequential_rewriter = corpus_rewriting.SequentialCorpusRewriter(
    rewriters=[chunking_rewriter, formatting_rewriter]
)
rewritten_corpus = ot.run(sequential_rewriter(toy_corpus))

print(f"Original docs: {len(toy_corpus)}")
print(f"Processed docs: {len(list(rewritten_corpus))}")
for doc in rewritten_corpus:
  print(f"Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

Original docs: 3
Processed docs: 6
Doc ID: lion_1
Content:
Title: Lion Facts
Facts: Lions are large cats that live in prides. They
--------------------
Doc ID: lion_2
Content:
Title: Lion Facts
Facts: They are apex predators.
--------------------
Doc ID: elephant_1
Content:
Title: Elephant Facts
Facts: Elephants are the largest land animals. They have
--------------------
Doc ID: elephant_2
Content:
Title: Elephant Facts
Facts: have long trunks and tusks.
--------------------
Doc ID: monkey_1
Content:
Title: Monkey Facts
Facts: Monkeys are primates that often live in trees.
--------------------
Doc ID: monkey_2
Content:
Title: Monkey Facts
Facts: . They are known for their intelligence.
--------------------


Observed the content for **`Chunk ID: lion_2`** now:

> *Content: Title: Lion Facts Facts: They are apex predators.*

Note: The order of chunking and formatting is not set in stone—you can easily
interchange them (e.g., formatting a document first, and then chunking it) depending on your use case. Furthermore,
`SequentialCorpusRewriter` can chain together **any number of custom processing steps**.

## Putting It All Together: The `RewritingIndex`

In the previous steps, we manually ran our raw toy documents through the Corpus
Rewriter so we could inspect the formatted chunks. However, in a
production application, you don't want to manually manage these intermediate
processed chunks.

To simplify this, OneTwo provides the `RewritingIndex` interface. This class
acts as a unified wrapper for the **entire** Index Building pipeline. It
packages your Corpus Rewriter (the Chunker and Formatter) and the Index into a
single, seamless object.

In [31]:
rewritten_index = indexing.RewritingEmbeddingBasedDocumentIndex(
    rewriter=sequential_rewriter
)

# Populate the index with our processed docs.
ot.run(rewritten_index.add_docs(toy_corpus))
print(f"Rewritten index created with {rewritten_index.num_docs} documents.")

Rewritten index created with 6 documents.


> NOTE: When you call `add_docs()`, you pass the **raw, unstructured Corpus**
> (`toy_corpus`). The `RewritingIndex` automatically processes it
> through the rewriter and populates the index with the resulting
> embeddings in one single step!

In [32]:
# Test retrieval on the rewritten index.
query = "Tell me about lions."
results = ot.run(rewritten_index.retrieve(query, max_results=4))

print(f"Query: {query}\n")
for doc in results:
  print(f"Retrieved Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

Query: Tell me about lions.

Retrieved Doc ID: lion_1
Content:
Title: Lion Facts
Facts: Lions are large cats that live in prides. They
--------------------
Retrieved Doc ID: lion_2
Content:
Title: Lion Facts
Facts: They are apex predators.
--------------------
Retrieved Doc ID: elephant_1
Content:
Title: Elephant Facts
Facts: Elephants are the largest land animals. They have
--------------------
Retrieved Doc ID: elephant_2
Content:
Title: Elephant Facts
Facts: have long trunks and tusks.
--------------------


Observe the output of our retrieval for the query *"Tell me about lions."*

The relevant lion facts are correctly retrieved at the top of the list, as
OneTwo retrievers automatically sort results by similarity score.

However, `max_results=4` forces the retriever to return exactly four documents
resulting in two irrelevant elephant facts at the bottom. In a production RAG
system, feeding this irrelevant noise to the LLM wastes context tokens and may
cause hallucinations.

We can inspect the embedding similarity scores using the
`retrieve_with_scores()` API.

In [33]:
# Retrieve documents alongside their similarity scores.
results_with_scores = ot.run(
    rewritten_index.retrieve_with_scores(query, max_results=4)
)
print(f"Query: {query}\n")
for doc, score in results_with_scores:
  print(f"Doc ID: {doc.doc_id} | Score: {score:.3f}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

Query: Tell me about lions.

Doc ID: lion_1 | Score: 0.763
Content:
Title: Lion Facts
Facts: Lions are large cats that live in prides. They
--------------------
Doc ID: lion_2 | Score: 0.740
Content:
Title: Lion Facts
Facts: They are apex predators.
--------------------
Doc ID: elephant_1 | Score: 0.633
Content:
Title: Elephant Facts
Facts: Elephants are the largest land animals. They have
--------------------
Doc ID: elephant_2 | Score: 0.626
Content:
Title: Elephant Facts
Facts: have long trunks and tusks.
--------------------


As seen above, the relevant lion documents have higher similarity scores,
whereas the elephant documents have lower scores. We can enforce a `min_score`
threshold to ensure that the retriever only returns documents that meet a
minimum score threshold, even if `max_results` is set to a higher value.

In [34]:
# Retrieve with both max_results and a minimum score threshold.
filtered_results = ot.run(
    rewritten_index.retrieve(query, max_results=4, min_score=0.7)
)
print(f"Query: {query} (with min_score=0.7)\n")
print(f"Retrieved {len(filtered_results)} documents.\n")
for doc in filtered_results:
  print(f"Retrieved Doc ID: {doc.doc_id}")
  print(f"Content:\n{doc.content}")
  print("-" * 20)

Query: Tell me about lions. (with min_score=0.7)

Retrieved 2 documents.

Retrieved Doc ID: lion_1
Content:
Title: Lion Facts
Facts: Lions are large cats that live in prides. They
--------------------
Retrieved Doc ID: lion_2
Content:
Title: Lion Facts
Facts: They are apex predators.
--------------------


# Question Answering (QA) Strategies

Now that we have built a structured Index that acts as our Retriever, we can
complete the RAG pipeline by performing Augmentation and Generation. In this
section, we will explore how to orchestrate the Index and the LLM to answer user
questions.

## QA Strategy: Retrieve and Summarize

The most common and intuitive RAG pattern is to retrieve relevant documents from
our Index and "stuff" them directly into the LLM's prompt as context. This is
called a **Retrieve-and-Summarize** strategy.

To construct these prompts cleanly, OneTwo provides `QADocumentPromptBuilder`,
which standardizes how the retrieved document chunks and the user's question are
formatted together.

In [35]:
from onetwo.stdlib.qa import qa_prompt_builder
from onetwo.stdlib.retrieval import retrieval_data_structures

fake_retrieved_docs = [
    retrieval_data_structures.Document(
        content="Lions are large cats that live in prides.", doc_id="doc1"
    ),
    retrieval_data_structures.Document(
        content="Elephants are the largest land animals.", doc_id="doc2"
    ),
]

default_prompt_builder = qa_prompt_builder.QADocumentPromptBuilder()
default_qa_prompt = ot.run(
    default_prompt_builder(fake_retrieved_docs, "Lions?")
)
print(default_qa_prompt.prefix)

Use the following pieces of context to answer the question at
the end. If you don't know the answer, just say that you don't know,
don't try to make up an answer.

Context:
Lions are large cats that live in prides.

Elephants are the largest land animals.

Question: Lions?
Answer:


The `QADocumentPromptBuilder` is highly customizable, giving you full control
over how the retrieved text is structured (e.g., using custom delimiters or
document formatters) to align with your LLM's prompt guidelines. For example:

In [36]:
custom_prompt_builder = qa_prompt_builder.QADocumentPromptBuilder(
    document_formatter=document_formatting.TextDocumentFormatter(
        format_str="Title: {doc_id}\nCustom Content: {text}",
    ),
    delimiter="\n\n ========== \n\n",
)

custom_qa_prompt = ot.run(
    custom_prompt_builder(fake_retrieved_docs, "What color are apples?")
)
print(custom_qa_prompt.prefix)

Use the following pieces of context to answer the question at
the end. If you don't know the answer, just say that you don't know,
don't try to make up an answer.

Context:
Title: doc1
Custom Content: Lions are large cats that live in prides.


Title: doc2
Custom Content: Elephants are the largest land animals.

Question: What color are apples?
Answer:


### Packaging the Pipeline: RetrieveAndSummarize

We can package the entire workflow — querying the Index, building the QA prompt, and calling the LLM — into a single component.

In OneTwo, we do this by defining a strategy class that inherits from
`SimpleRetrievalQAStrategy`.

In [ ]:
from onetwo.stdlib.qa import retrieval_qa


@dataclasses.dataclass
class RetrieveAndSummarize(retrieval_qa.SimpleRetrievalQAStrategy):
  """A simple retrieve and summarize strategy."""

  max_results: int | None = None
  prompt_builder: qa_prompt_builder.QADocumentPromptBuilder = dataclasses.field(
      default_factory=qa_prompt_builder.QADocumentPromptBuilder
  )
  print_retrieved_docs: bool = False

  @ot.make_executable(copy_self=False, non_copied_args=["retriever"])
  async def __call__(
      self,
      question: str,
      retriever: retrieval.Retriever[str, retrieval_data_structures.Document],
  ) -> str:
    docs = await retriever.retrieve(question, max_results=self.max_results)

    if self.print_retrieved_docs:
      print(f"Retrieved {len(docs)} documents:")
      for i, d in enumerate(docs):
        print(f"--- Document {i} ---")
        print(d.content)
        print("------------------")

    qa_prompt = await self.prompt_builder(docs, question)
    return await llm.chat(qa_prompt.to_messages())


# Instantiate the strategy.
qa_strategy = RetrieveAndSummarize()

### Testing the RAG Pipeline

Let's put our new RAG strategy to the test using **our rewriting index**! We
will ask two very different questions to observe how the system behaves:

1.  **A "grounded" query**: *"Where do monkeys usually live?"* The answer should
    be easily retrieved from our formatted monkey facts.
2.  **An "unanswerable" query**: *"What is the largest land animal and where can
    I find it in the zoo?"* Our index knows elephants are the largest land
    animals, but has no information about where they live in the zoo. A
    well-behaved RAG system should recognize this limitation and **avoid
    hallucinating**.

Let's test it with a query that can be answered from our documents.

In [38]:
query_1 = "Where do monkeys usually live?"
answer_1 = ot.run(qa_strategy(query_1, rewritten_index))

print(f"Query 1: {query_1}")
print(f"Answer 1: {answer_1}")

Query 1: Where do monkeys usually live?
Answer 1: Monkeys usually live in trees.


Now let's see how it handles a query where the required information is missing.

In [39]:
query_2 = "What is the largest land animal and where can I find it in the zoo?"
answer_2 = ot.run(qa_strategy(query_2, rewritten_index))

print(f"Query 2: {query_2}")
print(f"Answer 2: {answer_2}")

Query 2: What is the largest land animal and where can I find it in the zoo?
Answer 2: The largest land animal is the elephant. The provided text does not contain information about where to find them in a zoo.


As you can see, once `rewritten_index` is built, we can call
`ot.run(qa_strategy(query, rewritten_index))` for as many queries as we want.

### Decoupling Index Building and Serving

By design, index building and question answering phases are completely decoupled:

* **Index Building**: Index building in OneTwo is incredibly flexible — it can run quickly *online* (in-memory) for local testing, or scale up *offline* (using parallel pipelines like Apache Beam) for huge corpora. Once built, the index is **fully serializable**, allowing you to save it to disk and reload it instantly.

* **Question Answering**: Your live application simply **deserializes (loads)**
the pre-built index, performing a fast vector search to generate answers in
milliseconds.

To see this decoupling in action, we can serialize our already built `rewritten_index` to disk via `save()` and then deserialize it back into a new index via `load()`.

In [40]:
# Define a temporary path to save our index.
index_path = "/tmp/onetwo_zoo_index"

# 1. Save the built index to disk.
rewritten_index.save(index_path)
print(f"Index successfully saved to: {index_path}")

# 2. Create a brand new, empty index with the same rewriter configuration.
loaded_index = indexing.RewritingEmbeddingBasedDocumentIndex(
    rewriter=sequential_rewriter
)
print(f"New index doc count before loading: {loaded_index.num_docs}")

# 3. Load the saved state into the new index.
ot.run(loaded_index.load(index_path))
print(f"New index doc count after loading: {loaded_index.num_docs}")

# 4. Verify the loaded index by running query 1 again.
print(f"Query 1: {query_1}")
answer_1_loaded = ot.run(qa_strategy(query_1, loaded_index))
print(f"Answer 1: {answer_1_loaded}")

# Clean up the temporary directory.
shutil.rmtree(index_path)

Index successfully saved to: /tmp/onetwo_zoo_index
New index doc count before loading: 0
New index doc count after loading: 6
Query 1: Where do monkeys usually live?
Answer 1: Monkeys usually live in trees.


# QA Strategies on Real-World Datasets

Now that we have mastered the individual pieces of the RAG pipeline — **Index
Building** and **Orchestrated QA** — let's combine them in an end-to-end,
real-world application.

We will use [HotpotQA](https://hotpotqa.github.io/), a famous multi-hop question
answering dataset where answering a single question requires retrieving and
reasoning over multiple distinct Wikipedia paragraphs. This perfectly highlights
RAG's ability to synthesize information spread across multiple documents.

In this final section, we will:

1.  Load the HotpotQA dataset using OneTwo's Dataset Loaders.
2.  Build a semantic **Index** over the Wikipedia documents.
3.  Run our orchestrated **QA strategy**.
4.  Programmatically **evaluate** the quality of our RAG system using OneTwo's
    evaluation framework.

## Load HotpotQA

OneTwo provides `HotpotQADatasetLoader`, a convenient wrapper that loads
HotpotQA via TFDS and returns:

-   **Examples**: Dictionaries with `question` and `answer` keys.
-   **Documents**: `Document` objects representing the Wikipedia paragraphs
    associated with each example, used to build the index.

We create a `HotpotQADatasetLoader` with `max_number_of_examples=5` and `max_number_of_documents=50` to load only
the first 5 examples (and their associated documents) for a quick preview.

In [ ]:
from onetwo.evaluation import datasets

loader = datasets.HotpotQADatasetLoader(
    split='validation',
    max_number_of_examples=5,
    max_number_of_documents=50,
)

In [42]:
examples = ot.run(loader.load_examples())
print(f'Loaded {len(examples)} examples.\n')

for i, example in enumerate(examples):
  print(f'--- Example {i + 1} ---')
  print(f'  Question: {example["question"]}')
  print(f'  Answer:   {example["answer"]}')

Loaded 5 examples.

--- Example 1 ---
  Question: William Davis Ticknor, Sr. served on the board of a chemical and biotechnology company created in what year?
  Answer:   1919
--- Example 2 ---
  Question: The 2009 Fort Hood mass shooting was committed by who?
  Answer:   Nidal Hasan
--- Example 3 ---
  Question: A Talk is an EP by which South Korean singer and dancer?
  Answer:   Hyuna
--- Example 4 ---
  Question: What did Karan Kapoor's maternal grandfather deliver?
  Answer:   Shakespeare performances
--- Example 5 ---
  Question: Who was the First Prophert to the sixth Pharaoh of the Eighteenth Dynasty?
  Answer:   Khonsu


In [43]:
documents = ot.run(loader.load_documents())
print(f'Loaded {len(documents)} unique documents.\n')

for i, doc in enumerate(documents[:5]):
  print(f'--- Document {i + 1} ---')
  print(f'  Title:    {doc.title}')
  print(
      '  Content: '
      f' {doc.content[:200]}{"..." if len(doc.content) > 200 else ""}'
  )

Loaded 50 unique documents.

--- Document 1 ---
  Title:    Icos
  Content:  Icos Corporation (trademark ICOS) was an American biotechnology company and the largest biotechnology company in the U.S. state of Washington, before it was sold to Eli Lilly and Company in 2007.  It ...
--- Document 2 ---
  Title:    Commercial Solvents Corporation
  Content:  Commercial Solvents Corporation (CSC) was an American chemical and biotechnology company created in 1919.
--- Document 3 ---
  Title:    Pascal Brandys
  Content:  Pascal Brandys (born 30 November 1958 in Roanne) is a French engineer and entrepreneur.  He is a graduate of the École Polytechnique and received his M.S. in Economic Systems from Stanford University ...
--- Document 4 ---
  Title:    Daiichi Sankyo
  Content:  Daiichi Sankyo Company, Limited (第一三共株式会社 , Daiichi Sankyō Kabushiki-kaisha ) is a global pharmaceutical company and the second largest pharmaceutical company in Japan.  It achieved JPY 1,148.2 billio...
--- Document 5

> Note that Documents should not be expected to be in a order matching the order
> of the examples.

## Building the HotpotQA Index

Now that we have our document corpus, we need to index it to make it searchable.

In [44]:
# Create an index for the HotpotQA documents.
hotpot_index = indexing.EmbeddingBasedDocumentIndex()
ot.run(hotpot_index.add_docs(documents))

print(f"Indexed {len(documents)} processed documents.")

Indexed 50 processed documents.


> Because `HotpotQA` already parses the raw data into clean, paragraph-sized
> document blocks, **we don't need to apply a Corpus Rewriter here**. We can
> skip the rewriting phase and build our `EmbeddingBasedDocumentIndex` directly
> from the loaded documents!

## Running QA on HotpotQA

Let's test our `RetrieveAndSummarize` strategy on a HotpotQA question, using our
newly built `hotpot_index`.

We configure the strategy to print the retrieved documents to visually verify
that the correct context is being augmented to the LLM.

In [45]:
# Get the first example from HotpotQA.
example = examples[0]
query = example["question"]
ground_truth = example["answer"]

print(f"Question: {query}\n")
print(f"Ground Truth Answer: {ground_truth}\n")

# Perform QA on the first example.
qa_strategy_hotpot = RetrieveAndSummarize(
    max_results=5, print_retrieved_docs=True
)
answer = ot.run(qa_strategy_hotpot(query, hotpot_index))
print(answer)

Question: William Davis Ticknor, Sr. served on the board of a chemical and biotechnology company created in what year?

Ground Truth Answer: 1919

Retrieved 5 documents:
--- Document 0 ---
William Davis Ticknor, Sr. (January 11, 1881 – March 24, 1938) was president and chairman of the board of Commercial Solvents Corporation.  He was also president of Commercial Pigments Corporation.
------------------
--- Document 1 ---
Commercial Solvents Corporation (CSC) was an American chemical and biotechnology company created in 1919.
------------------
--- Document 2 ---
William Davis Ticknor I (August 6, 1810 – April 10, 1864) was an American publisher in Boston, Massachusetts, USA, and a founder of the publishing house Ticknor and Fields.
------------------
--- Document 3 ---
Sanofi Genzyme is an American biotechnology company based in Cambridge, Massachusetts.  Since its acquisition in 2011, Genzyme has been a fully owned subsidiary of Sanofi.  In 2010, Genzyme was the world’s third-largest 

## Evaluating RAG Quality

Now that we have inspected a single example, lets evaluate performance across a larger, representative dataset.

OneTwo provides an evaluation framework to run our strategy over our loaded
examples. We will use an LLM-based critic to judge whether the generated answers match the ground-truth answers semantically.

In [ ]:
# Define a wrapper strategy for evaluation that takes only the question.
@ot.make_executable
async def qa_wrapper(question: str) -> str:
  # We disable printing retrieved docs for evaluation to avoid cluttering the output.
  strategy = RetrieveAndSummarize(print_retrieved_docs=False, max_results=5)
  return await strategy(question, hotpot_index)

In [47]:
from onetwo.evaluation import evaluation

print("Running evaluation on the first 5 examples...")
summary = ot.evaluate(
    strategy=qa_wrapper,
    examples=examples[:5],
    metric_functions={
        "semantic_critic_match": evaluation.naive_evaluation_critic
    },
    output_results=True,
)

# Print aggregated metrics.
print("\nAggregated Metrics:")
for metric_name, value in summary.metrics.items():
  print(f" - {metric_name}: {value}")

# Print detailed results.
print("\nDetailed Per-Example Results:")
for key, evaluation_result in summary.results.items():
  predicted_answer = evaluation_result.outputs.get("output", "N/A")
  target = evaluation_result.targets.get("target", "N/A")
  correctness = evaluation_result.metrics.get("semantic_critic_match", 0.0)
  info = evaluation_result.info
  reason = info.get("critic_reason", "N/A")
  print(f"Example {key}:")
  print(f"  Prediction: {predicted_answer}")
  print(f"  Target: {target}")
  print(f"  Score: {correctness}")
  print(f"  Reason: {reason}")

Running evaluation on the first 5 examples...


100%|██████████| 5/5 [00:00<00:00, 51.48it/s]


Aggregated Metrics:
 - semantic_critic_match: 0.8

Detailed Per-Example Results:
Example 0:
  Prediction: Answer: 1919
  Target: 1919
  Score: 1.0
  Reason: The prediction correctly identifies the year 1919, which matches the target answer. The inclusion of the word "Answer:" does not change the factual accuracy of the response.
Example 1:
  Prediction: The 2009 Fort Hood mass shooting was committed by Nidal Malik Hasan, a U.S. Army major and psychiatrist.
  Target: Nidal Hasan
  Score: 1.0
  Reason: The prediction correctly identifies the perpetrator (Nidal Malik Hasan) and provides additional accurate context, which does not contradict the target answer.
Example 2:
  Prediction: A Talk is an EP by the South Korean singer Hyuna.
  Target: Hyuna
  Score: 1.0
  Reason: The prediction correctly identifies Hyuna as the artist, providing the same information as the target answer in a complete sentence.
Example 3:
  Prediction: Karan Kapoor's maternal grandfather, Geoffrey Kendal, was an a

# OneTwo RAG in the Agent Landscape

In today's AI engineering landscape, **Agents**: autonomous systems that
leverage planning, memory, and tools to solve complex tasks, are highly
prominent. However, an agent is only as good as the information it can access, i.e. the best agents require a state-of-the-art **RAG system** to ground its reasoning, retrieve precise facts, and prevent hallucinations. OneTwo
RAG is built with this exact synergy in mind. Rather than forcing you into a
rigid, all-in-one agent framework, OneTwo focuses on providing a **highly
modular, best-in-class RAG toolbox** that is complementary to any agent
workflow.

## Agent-Agnostic OneTwo RAG

Because OneTwo's `Retriever` and `Index` abstractions are highly standardized, **the custom Index Retriever you built in this tutorial can be plugged directly as a Tool into any Agent architecture.**

To illustrate this complementarity, here is a quick, didactic example of how easily our animal index retriever can be wrapped as a standard `Tool` and handed to an autonomous `ReActAgent`:

In [ ]:
from onetwo.stdlib.tool_use import llm_tool_use


# Define the retrieval function.
def retrieve_zoo_info(query: str) -> str:
  """Retrieves facts about animals in the zoo."""
  # We use the rewritten_index we created earlier.
  docs = ot.run(rewritten_index.retrieve(query, max_results=4))
  return "\n".join([f"- {d.content}" for d in docs])


# Wrap it as a Tool.
zoo_retrieval_tool = llm_tool_use.Tool(
    name="retrieve_zoo_info",
    function=retrieve_zoo_info,
    description="Retrieves facts about animals in the zoo based on a query.",
    example="retrieve_zoo_info('elephant')",
)

To ensure the Agent understands how to use our custom Index tool, we provide it
with an exemplar (a few-shot example of a successful
"Thought-Action-Observation" loop).

In [ ]:
from onetwo.agents import react

# Define a custom exemplar for the ReAct agent to help it understand the format
# and how to use our specific tool.
custom_exemplar = react.ReActState(
    inputs="What does the lion eat?",
    updates=[
        react.ReActStep(
            thought=(
                "I need to find out what the lion eats. I can use the"
                " retrieve_zoo_info tool."
            ),
            action=llm_tool_use.FunctionCall(
                function_name="retrieve_zoo_info",
                args=("lion diet",),
                kwargs={},
            ),
            observation="Lions are carnivores and eat meat.",
            fmt=llm_tool_use.ArgumentFormat.PYTHON,
        ),
        react.ReActStep(
            is_finished=True,
            thought="I found that lions eat meat.",
            action=None,
            observation="Meat",
            fmt=llm_tool_use.ArgumentFormat.PYTHON,
        ),
    ],
)

Finally, we configure and run the `ReActAgent` with our tool and exemplar.

In [54]:
from onetwo.stdlib.tool_use import python_tool_use

# Configure the ReAct Agent.
config = python_tool_use.PythonToolUseEnvironmentConfig(
    tools=[zoo_retrieval_tool]
)

# We pass our custom exemplar to guide the agent.
agent = react.ReActAgent(
    environment_config=config,
    max_steps=5,
    exemplars=[custom_exemplar],
)

# Run the agent and request the final state.
query = "Are monkeys primates?"
output, final_state = ot.run(agent(inputs=query, return_final_state=True))

print(f"Query: {query}\n")
print(f"Final Output: {output}\n")

print("Execution Trace:")
for i, step in enumerate(final_state.updates):
  print(f"\nStep {i+1}:")
  if step.thought:
    print(f"Thought: {step.thought}")
  if step.is_finished:
    print(f"Final Answer: {step.observation}")
    break
  if step.action:
    print(f"Action: {step.action.function_name}({step.action.args})")
  if step.observation:
    print(f"Observation:\n {step.observation}")

Query: Are monkeys primates?

Final Output: Yes, monkeys are primates.

Execution Trace:

Step 1:
Thought: I need to determine if monkeys are classified as primates. I can use the retrieve_zoo_info tool to check their classification.
Action: retrieve_zoo_info(('monkey classification',))
Observation:
 - Title: Monkey Facts
Facts: Monkeys are primates that often live in trees.
- Title: Monkey Facts
Facts: . They are known for their intelligence.
- Title: Lion Facts
Facts: They are apex predators.
- Title: Elephant Facts
Facts: have long trunks and tusks.

Step 2:
Thought: The retrieved information confirms that monkeys are primates.
Final Answer: Yes, monkeys are primates.


In [ ]:
cached_backends.save_caches()